# Advanced NLP Processing
## Advanced natural language processing techniques and analysis

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import re
from typing import List, Dict, Tuple
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tag import pos_tag
from nltk.chunk import ne_chunk
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

## Download NLTK Data

In [ ]:
# Download required NLTK data
try:
    nltk.data.find('sentiment/vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('maxent_ne_chunker', quiet=True)
    nltk.download('words', quiet=True)

print("✓ NLTK data ready")

## TextAnalyzer Class

In [ ]:
class TextAnalyzer:
    """Advanced text analysis and NLP processing"""
    
    def __init__(self):
        """Initialize analyzer with NLTK components"""
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        self.sentiment_analyzer = SentimentIntensityAnalyzer()
    
    def analyze_text(self, text: str) -> Dict:
        """Comprehensive text analysis"""
        if pd.isna(text) or not isinstance(text, str):
            return {}
        
        text_lower = text.lower()
        
        # Basic statistics
        chars = len(text)
        words = len(text.split())
        sentences = len(sent_tokenize(text))
        
        # Tokenization and cleanup
        tokens = word_tokenize(text_lower)
        clean_tokens = [t for t in tokens if t.isalnum() and t not in self.stop_words]
        unique_words = len(set(clean_tokens))
        
        # Lexical diversity
        lexical_diversity = unique_words / len(clean_tokens) if clean_tokens else 0
        
        # Average word length
        avg_word_length = np.mean([len(w) for w in clean_tokens]) if clean_tokens else 0
        
        # Sentiment analysis
        sentiment = self.sentiment_analyzer.polarity_scores(text)
        
        return {
            'text_length': chars,
            'word_count': words,
            'sentence_count': sentences,
            'unique_words': unique_words,
            'lexical_diversity': lexical_diversity,
            'avg_word_length': avg_word_length,
            'avg_sentence_length': words / sentences if sentences > 0 else 0,
            'sentiment_score': sentiment['compound'],
            'sentiment_label': self._get_sentiment_label(sentiment['compound'])
        }
    
    @staticmethod
    def _get_sentiment_label(score: float) -> str:
        """Convert sentiment score to label"""
        if score >= 0.05:
            return 'Positive'
        elif score <= -0.05:
            return 'Negative'
        else:
            return 'Neutral'
    
    def extract_keywords(self, text: str, n_keywords: int = 10) -> List[str]:
        """Extract top keywords from text"""
        if pd.isna(text):
            return []
        
        tokens = word_tokenize(str(text).lower())
        keywords = [t for t in tokens if t.isalnum() and t not in self.stop_words]
        counter = Counter(keywords)
        return [word for word, _ in counter.most_common(n_keywords)]

## TextSimilarity Class

In [ ]:
class TextSimilarity:
    """Calculate text similarity measures"""
    
    @staticmethod
    def jaccard_similarity(text1: str, text2: str) -> float:
        """Calculate Jaccard similarity between two texts"""
        if pd.isna(text1) or pd.isna(text2):
            return 0
        
        tokens1 = set(str(text1).lower().split())
        tokens2 = set(str(text2).lower().split())
        
        intersection = len(tokens1.intersection(tokens2))
        union = len(tokens1.union(tokens2))
        
        return intersection / union if union > 0 else 0
    
    @staticmethod
    def edit_distance(text1: str, text2: str) -> int:
        """Calculate edit distance (Levenshtein distance) between two texts"""
        if pd.isna(text1) or pd.isna(text2):
            return 0
        
        text1 = str(text1)
        text2 = str(text2)
        
        m, n = len(text1), len(text2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        for i in range(m + 1):
            dp[i][0] = i
        for j in range(n + 1):
            dp[0][j] = j
        
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if text1[i-1] == text2[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
        
        return dp[m][n]
    
    @staticmethod
    def normalized_edit_distance(text1: str, text2: str) -> float:
        """Calculate normalized edit distance (0-1)"""
        distance = TextSimilarity.edit_distance(text1, text2)
        max_length = max(len(str(text1)), len(str(text2)))
        return distance / max_length if max_length > 0 else 0

## BatchTextProcessor Class

In [ ]:
class BatchTextProcessor:
    """Process multiple texts in batch"""
    
    def __init__(self):
        """Initialize batch processor"""
        self.analyzer = TextAnalyzer()
        self.similarity = TextSimilarity()
    
    def analyze_corpus(self, texts: List[str]) -> pd.DataFrame:
        """Analyze multiple texts and return summary"""
        results = []
        
        for text in texts:
            analysis = self.analyzer.analyze_text(text)
            analysis['original_text'] = text[:100]
            results.append(analysis)
        
        df = pd.DataFrame(results)
        return df
    
    def extract_corpus_keywords(self, texts: List[str], n_keywords: int = 20) -> Dict[str, int]:
        """Extract top keywords from corpus"""
        all_keywords = []
        
        for text in texts:
            keywords = self.analyzer.extract_keywords(text, n_keywords=100)
            all_keywords.extend(keywords)
        
        counter = Counter(all_keywords)
        return dict(counter.most_common(n_keywords))
    
    def find_similar_texts(self, query: str, texts: List[str], 
                          method: str = 'jaccard', top_n: int = 5) -> List[Tuple[str, float]]:
        """Find texts similar to query"""
        similarities = []
        
        for text in texts:
            if method == 'jaccard':
                sim = self.similarity.jaccard_similarity(query, text)
            else:
                sim = self.similarity.normalized_edit_distance(query, text)
            
            similarities.append((text, sim))
        
        if method == 'edit_distance':
            similarities.sort(key=lambda x: x[1])
        else:
            similarities.sort(key=lambda x: x[1], reverse=True)
        
        return similarities[:top_n]

## Example Usage

In [ ]:
# Example texts
example_texts = [
    "Machine learning is revolutionizing artificial intelligence applications",
    "Natural language processing enables computers to understand human language",
    "Deep learning models have achieved state-of-the-art results",
    "Data science combines statistics and programming",
]

# Initialize processors
analyzer = TextAnalyzer()
batch_processor = BatchTextProcessor()

print("✓ NLP processors initialized")

## Single Text Analysis

In [ ]:
# Analyze single text
text = example_texts[0]
print(f"\nAnalyzing: {text}")
print("="*60)

analysis = analyzer.analyze_text(text)
for key, value in analysis.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

## Keyword Extraction

In [ ]:
# Extract keywords
keywords = analyzer.extract_keywords(text, n_keywords=5)
print(f"\nTop keywords: {keywords}")

## Batch Analysis

In [ ]:
# Batch analysis
print("\nBatch Text Analysis")
print("="*60)
corpus_df = batch_processor.analyze_corpus(example_texts)
corpus_df[['word_count', 'sentiment_label', 'lexical_diversity']]

## Corpus Keywords

In [ ]:
# Corpus keywords
corpus_keywords = batch_processor.extract_corpus_keywords(example_texts, n_keywords=10)
print("\nCorpus Keywords:")
print("="*60)
for keyword, count in corpus_keywords.items():
    print(f"{keyword}: {count}")

## Text Similarity

In [ ]:
# Similarity
query = "machine learning and artificial intelligence"
print(f"\nQuery: {query}")
print("="*60)

similar = batch_processor.find_similar_texts(query, example_texts, method='jaccard', top_n=3)
print("\nMost Similar Texts:")
for i, (text, similarity) in enumerate(similar, 1):
    print(f"{i}. Similarity: {similarity:.4f}")
    print(f"   Text: {text[:70]}...")
    print()